In [1]:
# Project paths: repo root + src/ on sys.path (run with cwd = repository root)
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    # Fallback if the notebook kernel cwd is notebooks/
    candidate = PROJECT_ROOT.parent
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

import earthaccess
import xarray as xr
import numpy as np
import dask
import glob
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
%matplotlib inline
import geopandas as gpd
from cftime import num2date
from dask.diagnostics import ProgressBar

from grace_plot_eda import (
    plot_geo_map,
    plot_map_with_distribution,
)

# Auto-reloading setup for utility functions
%load_ext autoreload
%autoreload 2

dask.config.set(scheduler='processes', num_workers=12)


# Download GPM IMERG Data

How to Read IMERG Data Using Python

Prerequisites

This notebook was written using Python 3.10, and requires:
- Valid [Earthdata Login credentials](https://urs.earthdata.nasa.gov), and the generation of [Earthdata Prerequisite Files](https://disc.gsfc.nasa.gov/information/howto?title=How%20to%20Generate%20Earthdata%20Prerequisite%20Files) including the <code>.netrc</code> and `.dodsrc` files.
- [Xarray](https://docs.xarray.dev/en/stable/)
- [earthaccess](https://earthaccess.readthedocs.io/en/latest/)
- [NumPy (v1.26)](https://numpy.org/)
- [Matplotlib](https://matplotlib.org/)
- [Cartopy](https://scitools.org.uk/cartopy/docs/latest/)

## Search, download and process monthly data

In [ ]:
%%time
# Authenticate with Earthdata Login servers
auth = earthaccess.login()

# Search for granules
results_m = earthaccess.search_data(
    short_name="GPM_3IMERGM",  # change the product
    version="07",
    temporal=('1998-01-01', '2024-06-30'),
    bounding_box=(-180, 0, 180, 90),
    count=-1
)

# Print search results
print(len(results_m))

In [ ]:
%%time
# Download the granule to the current working directory
downloaded_files_m = earthaccess.download(
    results_m,
    local_path=str(RAW_DIR / "gpm" / "GPM_3IMERGM"), # Change this string to download to a different path
    threads = 16
)


Stream the dataset to a variable and display metadata

In [ ]:
%%time
# Load all variables minimally
ds_m = xr.open_mfdataset(downloaded_files_m, group="Grid")

# Subset the dataset to include only precipitation and randomError
ds_m = ds_m[["precipitation"]]
#convert mm/h to mm/month
conversion_factor = 24 * 30
ds_m['precipitation'] *= conversion_factor

In [ ]:
# Update the 'EndDate' attribute to the most recent date in the dataset
end_date = str(ds_m.time.values[-1])[:10]  # Get the last time value and format as YYYY-MM-DD
ds_m.attrs['EndDate'] = end_date

# Remove the 'InputPointer' attribute
if 'InputPointer' in ds_m.attrs:
    del ds_m.attrs['InputPointer']

In [ ]:
%%time
# Save the dataset to a Zarr file
#time= 2 minutes
zarr_path_m = str(INTERIM_DIR / "gpm" / "GPM_3IMERGM_Jan1998_Jun2024.zarr")
ds_m.to_zarr(zarr_path_m, mode="w")  # Use mode="w" to overwrite if it exists


In [ ]:
# Get the precipitation, latitude, and longitude variables
precip = ds['precipitation'].sel(time='2018-05').squeeze().values
precip = np.transpose(precip)
theLats = ds['lat'].values
theLons = ds['lon'].values
x, y = np.float32(np.meshgrid(theLons, theLats))

In [ ]:
precip_da = ds['precipitation'].sel(time='2018-05').squeeze()

plot_geo_map(
    precip_da,
    title='GPM IMERG Monthly Mean Rain Rate for January 2014',
    cmap='rainbow',
    tick_fontsize=16,
    label_fontsize=18,
    title_fontsize=22,
    vmin=0,
    vmax=3,
    colorbar_extend='max',
)
plt.show()

Save the figure as a PNG:

In [ ]:
fig.savefig('GPM_3IMERG_plot.png', bbox_inches='tight', pad_inches = 0.1)

## Search, download, and process daily data

In [ ]:
# Authenticate with Earthdata Login servers
auth = earthaccess.login()

# Search for granules
results_d = earthaccess.search_data(
    short_name="GPM_3IMERGDF",  # change the product
    version="07",
    temporal=('2025-06-01', '2025-09-30'),
    bounding_box=(-180, 0, 180, 90),
    count=-1
)

# Print search results
print(len(results_d))

In [ ]:
%%time
# time=50 min
# Download the granule to the current working directory
downloaded_files_d = earthaccess.download(
    results_d,
    local_path=str(RAW_DIR / "gpm" / "GPM_3IMERGDF"), # Change this string to download to a different path
    threads = 16
)


Stream the dataset to a variable and display metadata

In [ ]:
#if you want to get all data in a directory

downloaded_files_d = sorted(glob.glob(str(RAW_DIR / "gpm" / "GPM_3IMERGDF" / "*.nc4")))
print(downloaded_files_d)  # show the first few to confirm


In [ ]:
%%time
# Load all files
#time= 15 min
ds_d = xr.open_mfdataset(
    downloaded_files_d,
    drop_variables=["time_bnds"],
    data_vars="minimal",
    coords="minimal",
    compat="override")

# Subset the dataset to include only precipitation and randomError
ds_d = ds_d[["precipitation"]]

In [ ]:
# Update the 'EndDate' attribute to the most recent date in the dataset
end_date = str(ds_d.time.values[-1])[:10]  # Get the last time value and format as YYYY-MM-DD
ds_d.attrs['EndDate'] = end_date

# Remove the 'InputPointer' attribute
if 'InputPointer' in ds_d.attrs:
    del ds_d.attrs['InputPointer']

In [ ]:
%%time
# Save the dataset to a Zarr file
zarr_path_d = str(INTERIM_DIR / "gpm" / "GPM_3IMERGDF_Jan1998_Sep2025.zarr")
ds_d.to_zarr(zarr_path_d, mode="w")  # Use mode="w" to overwrite if it exists


In [ ]:
%%time
#save zarr file of GPM IMERG Final precipitation daily resampled to monthly.
resampled_daily = ds_d.resample(time='ME').sum()
zarr_path_d_to_m = str(INTERIM_DIR / "gpm" / "GPM_3IMERGDF_Jan1998_Sep2025_resToM.zarr")
resampled_daily.to_zarr(zarr_path_d_to_m, mode="w")  # Use mode="w" to overwrite if it exists


# Download GLDAS-LSM Data

In [11]:
%%time
# Authenticate with Earthdata Login servers
auth = earthaccess.login()

#model = "GLDAS_NOAH10_M"
model = "GLDAS_CLSM10_M"
model = "GLDAS_VIC10_M"

# Search for granules
results_m = earthaccess.search_data(
    short_name=model,  # change the product
    version="2.1",
    temporal=('2000-01', '2026-1'),
    bounding_box=(-180, 0, 180, 90),
    count=-1
)

# Print search results
print(len(results_m))

313
CPU times: user 39 ms, sys: 9.38 ms, total: 48.4 ms
Wall time: 1.64 s


In [12]:
%%time
# Download the granule to the current working directory
local_path=str(RAW_DIR / "gldas" / model)
downloaded_files_m = earthaccess.download(
    results_m,
    local_path=local_path, # Change this string to download to a different path
    threads = 10
)

downloaded_files_m = sorted(glob.glob(local_path +"/*.nc4"))
len(downloaded_files_m)


QUEUEING TASKS | :   0%|          | 0/313 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/313 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/313 [00:00<?, ?it/s]

CPU times: user 65.5 ms, sys: 70 ms, total: 135 ms
Wall time: 111 ms


313

Stream the dataset to a variable and display metadata

Updates:
- NOAH SWE is done
- 

In [13]:
%%time
# Define the list of variables to extract
# variables = [
 
#     "SoilMoi0_10cm_inst", "SoilMoi100_200cm_inst", 
#     "SoilMoi10_40cm_inst", "SoilMoi40_100cm_inst"
# ]

#variables = ["SoilMoist_P_inst"] 

#variables = ["SoilMoi0_30cm_inst", "SoilMoi_depth2_inst", "SoilMoi_depth3_inst"]

#variables = ["Qs_acc", "Qsb_acc"]

variables = ["SWE_inst"]


# Open the dataset and select the specific variables
ds_m = xr.open_mfdataset(downloaded_files_m, engine="h5netcdf")[variables]
ds_m

CPU times: user 2min 38s, sys: 1.26 s, total: 2min 39s
Wall time: 3min 12s


<xarray.Dataset> Size: 68MB
Dimensions:   (time: 313, lat: 150, lon: 360)
Coordinates:
  * time      (time) datetime64[ns] 3kB 2000-01-01 2000-02-01 ... 2026-01-01
  * lon       (lon) float32 1kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * lat       (lat) float32 600B -59.5 -58.5 -57.5 -56.5 ... 86.5 87.5 88.5 89.5
Data variables:
    SWE_inst  (time, lat, lon) float32 68MB dask.array<chunksize=(1, 150, 360), meta=np.ndarray>
Attributes: (12/19)
    CDI:                    Climate Data Interface version 1.9.8 (https://mpi...
    Conventions:            CF-1.6
    history:                created on date: 2020-02-03T12:00:52.166
    source:                 VIC_v4.1.2 forced with GDAS-AGRMET-GPCPv13rA1
    institution:            NASA GSFC
    missing_value:          -9999.0
    ...                     ...
    MAP_PROJECTION:         EQUIDISTANT CYLINDRICAL
    SOUTH_WEST_CORNER_LAT:  -59.5
    SOUTH_WEST_CORNER_LON:  -179.5
    DX:                     1.0
    DY:                     1.0
    CDO:                    Climate Data Operators version 1.9.8 (https://mpi...

In [ ]:
# ds_m = ds_m.rename({"X": "lon", "Y": "lat"})
# ds_m.rio.set_spatial_dims(y_dim="lat", x_dim="lon", inplace=True)
# ds_m = ds_m.transpose("time", "lat", "lon")

In [ ]:
# Sum soil moisture layers while keeping the time dimension
# ds_m["sm_total"] = (
#     ds_m["SoilMoi0_10cm_inst"] +
#     ds_m["SoilMoi10_40cm_inst"] +
#     ds_m["SoilMoi40_100cm_inst"] +
#     ds_m["SoilMoi100_200cm_inst"]
# )

ds_m["sm_total"] = (
    ds_m["SoilMoi0_30cm_inst"] +
    ds_m["SoilMoi_depth2_inst"] +
    ds_m["SoilMoi_depth3_inst"]
)

# ds_m["total_runoff"] = (
#     ds_m["Qs_acc"] +
#     ds_m["Qsb_acc"]
# )


In [14]:
%%time
# Define the output path
zarr_path_m = str(INTERIM_DIR / "gldas" / model / f"{model}_SWE_Jan2000_Jan2026.zarr")

# Ensure dataset uses optimal chunking
ds_chunked = ds_m.chunk({"time": 100})  # Adjust based on available memory
ds_chunked


CPU times: user 5.85 ms, sys: 0 ns, total: 5.85 ms
Wall time: 5.43 ms


<xarray.Dataset> Size: 68MB
Dimensions:   (time: 313, lat: 150, lon: 360)
Coordinates:
  * time      (time) datetime64[ns] 3kB 2000-01-01 2000-02-01 ... 2026-01-01
  * lon       (lon) float32 1kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * lat       (lat) float32 600B -59.5 -58.5 -57.5 -56.5 ... 86.5 87.5 88.5 89.5
Data variables:
    SWE_inst  (time, lat, lon) float32 68MB dask.array<chunksize=(100, 150, 360), meta=np.ndarray>
Attributes: (12/19)
    CDI:                    Climate Data Interface version 1.9.8 (https://mpi...
    Conventions:            CF-1.6
    history:                created on date: 2020-02-03T12:00:52.166
    source:                 VIC_v4.1.2 forced with GDAS-AGRMET-GPCPv13rA1
    institution:            NASA GSFC
    missing_value:          -9999.0
    ...                     ...
    MAP_PROJECTION:         EQUIDISTANT CYLINDRICAL
    SOUTH_WEST_CORNER_LAT:  -59.5
    SOUTH_WEST_CORNER_LON:  -179.5
    DX:                     1.0
    DY:                     1.0
    CDO:                    Climate Data Operators version 1.9.8 (https://mpi...

In [15]:
%%time
#Expected time 30 hours, took 26.5 hours
# Enable a progress bar
with ProgressBar():
    with dask.config.set(scheduler="threads"):  # Use "processes" for heavy CPU workloads
        ds_chunked.to_zarr(zarr_path_m, mode="w", consolidated=True)
ds_chunked

[########################################] | 100% Completed | 30.25 s
CPU times: user 16.2 s, sys: 1.97 s, total: 18.1 s
Wall time: 31.3 s


<xarray.Dataset> Size: 68MB
Dimensions:   (time: 313, lat: 150, lon: 360)
Coordinates:
  * time      (time) datetime64[ns] 3kB 2000-01-01 2000-02-01 ... 2026-01-01
  * lon       (lon) float32 1kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * lat       (lat) float32 600B -59.5 -58.5 -57.5 -56.5 ... 86.5 87.5 88.5 89.5
Data variables:
    SWE_inst  (time, lat, lon) float32 68MB dask.array<chunksize=(100, 150, 360), meta=np.ndarray>
Attributes: (12/19)
    CDI:                    Climate Data Interface version 1.9.8 (https://mpi...
    Conventions:            CF-1.6
    history:                created on date: 2020-02-03T12:00:52.166
    source:                 VIC_v4.1.2 forced with GDAS-AGRMET-GPCPv13rA1
    institution:            NASA GSFC
    missing_value:          -9999.0
    ...                     ...
    MAP_PROJECTION:         EQUIDISTANT CYLINDRICAL
    SOUTH_WEST_CORNER_LAT:  -59.5
    SOUTH_WEST_CORNER_LON:  -179.5
    DX:                     1.0
    DY:                     1.0
    CDO:                    Climate Data Operators version 1.9.8 (https://mpi...